In [14]:
from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from datetime import datetime

from tools import (
    get_current_time,
    tool_schema
)

import faiss
import numpy as np

In [15]:
pdf_path = "sample_data/sample.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    extracted_text = page.extract_text()

    if extracted_text:
        text += extracted_text

print("PDF Loaded Successfully")
print(text[:500])

PDF Loaded Successfully
Basics of Computer :: 1
 
Basics of Computer
1.1 INTRODUCTION
In this lesson we present an overview of the basic design of a
computer system: how the different parts of a computer system
are organized and various operations performed to perform a
specific task. You would have observed that instructions have to
be fed into the computer in a systematic order to perform a
specific task. Computer components are divided into two major
categories, namely, hardware and software. In this lesson we will



In [16]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print("Total Chunks:", len(chunks))

Total Chunks: 50


In [17]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/home/nineleaps/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [18]:
embeddings = embedding_model.encode(chunks)

print(embeddings.shape)

(50, 384)


In [19]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS Index Created")

FAISS Index Created


In [20]:
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

In [21]:
chat_history = []

In [22]:
def rewrite_question(question, history):

    followup_words = [
        "it",
        "this",
        "that",
        "more",
        "explain more",
        "tell me more"
    ]

    if len(history) == 0:
        return question

    lower_question = question.lower()

    for word in followup_words:

        if word in lower_question:

            previous_user_questions = [
                msg["content"]
                for msg in history
                if msg["role"] == "user"
            ]

            if previous_user_questions:
                previous_question = previous_user_questions[-1]

                return (
                    previous_question
                    + " "
                    + question
                )

    return question

In [23]:
def should_call_tool(question):

    tool_keywords = [
        "time",
        "current time",
        "date",
        "clock"
    ]

    question = question.lower()

    for keyword in tool_keywords:

        if keyword in question:
            return True

    return False

In [26]:
def retrieve_context(question):

    question_embedding = embedding_model.encode(
        [question]
    )

    distances, indices = index.search(
        np.array(question_embedding),
        3
    )

    retrieved_chunks = [
        chunks[i]
        for i in indices[0]
    ]

    context = "\n".join(
        retrieved_chunks
    )

    return context, distances[0][0]

In [27]:
while True:

    question = input(
        "\nYou: "
    )

    if question.lower() == "exit":
        print("Assistant: Goodbye!")
        break

    # Step 1
    standalone_question = rewrite_question(
        question,
        chat_history
    )

    print(
        "\nStandalone Question:",
        standalone_question
    )

    # Step 2
    if should_call_tool(
        standalone_question
    ):

        tool_result = (
            get_current_time()
        )

        answer = (
            f"The current time is {tool_result}"
        )

        print("\n[TOOL CALLED]")
        print(
            "Tool Output:",
            tool_result
        )

    else:

        # Step 3
        context, score = retrieve_context(
            standalone_question
        )

        print("\nRetrieved Context:")
        print(context[:500])

        # Step 4
        prompt = f"""
        You are a helpful assistant.

        Answer ONLY using the context.

        Context:
        {context}

        Question:
        {standalone_question}

        Answer:
        """

        response = llm(
            prompt,
            max_new_tokens=150
        )

        answer = response[0][
            "generated_text"
        ]

    # Step 5
    print("\nAssistant:")
    print(answer)

    # Step 6
    chat_history.append(
        {
            "role": "user",
            "content": question
        }
    )

    chat_history.append(
        {
            "role": "assistant",
            "content": answer
        }
    )


You:  what is computer



Standalone Question: what is computer

Retrieved Context:
software
/circle6describe computer language and its classification
1
2 :: Data Entry Operations
1.3 WHAT IS A COMPUTER?
Computer is a device that transforms data into meaningful
information. Data can be anything like marks obtained by you in
various subjects. It can also be name, age, sex, weight, height,
etc. of all the students in a class.
Computer can also be defined in terms of functions it can
perform. A computer can i) accept data, ii) store data, iii) process
Basics of Computer :: 1
 
Bas

Assistant:
a device that transforms data into meaningful information



You:  more about it



Standalone Question: what is computer more about it

Retrieved Context:
Basics of Computer :: 1
 
Basics of Computer
1.1 INTRODUCTION
In this lesson we present an overview of the basic design of a
computer system: how the different parts of a computer system
are organized and various operations performed to perform a
specific task. You would have observed that instructions have to
be fed into the computer in a systematic order to perform a
specific task. Computer components are divided into two major
categories, namely, hardware and software. In this lesson we will


Assistant:
a device that transforms data into meaningful information



You:  current date and time



Standalone Question: current date and time

[TOOL CALLED]
Tool Output: 2026-06-02 16:29:56

Assistant:
The current time is 2026-06-02 16:29:56



You:  exit


Assistant: Goodbye!
